# 19 — Extraction attributaire et spatiale avec bilan

**Objectif :** préparer plusieurs critères, comparer ET et OU, expliquer les exclusions et livrer deux couches vérifiées. Les observations ci-dessous sont entièrement **simulées** ; elles ne décrivent aucun terrain réel.

La sélection conserve les objets entiers. Elle ne découpe pas les géométries. Chaque ligne source appartient exactement à l’une des deux partitions. [Contrat G07](../../docs/VECTOR_SELECTION.md).

In [1]:
from datetime import date
import hashlib
import json
from pathlib import Path
import sqlite3
from threading import Event
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, box
import cartomize as cm
from cartomize.vector_io import write_geopackage
from cartomize.viewer_data import table_page
from course import run_directory

work = run_directory('19_extraction')
assert tuple(map(int, cm.__version__.split('.')[:2])) >= (1, 10)
print('Moteur commun :', cm.__version__)

Moteur commun : 1.10.0


## 1. Une source dont chaque cas est connu

Les grands identifiants dépassent la précision exacte d’un flottant de 64 bits. Le code `NA` est un vrai texte. Une géométrie manque ; une autre est sur la frontière. Les dates sont des dates natives.

In [2]:
source = gpd.GeoDataFrame({
    'code': pd.Series(['001', 'NA', '003', '004', '005', '006'], dtype='string'),
    'identifiant': pd.Series([2**53+i for i in range(1, 7)], dtype='Int64'),
    'actif': pd.Series([True, True, False, False, None, True], dtype='boolean'),
    'visite': [date(2026,1,1), date(2026,2,1), None, date(2026,3,1), None, date(2026,1,10)],
}, geometry=[Point(1,1), Point(5,5), Point(2,2), Point(6,6), None, Point(0,1)], crs=32733)
# Deux polygones identiques ne doivent pas dupliquer les lignes.
mask = gpd.GeoDataFrame(geometry=[box(0,0,3,3), box(0,0,3,3)], crs=source.crs)
original_wkb = source.geometry.to_wkb().tolist()
source[['code', 'identifiant', 'actif', 'visite']]

,code,identifiant,actif,visite
0,001,9007199254740993,True,2026-01-01
1,NA,9007199254740994,True,2026-02-01
2,003,9007199254740995,False,None
3,004,9007199254740996,False,2026-03-01
4,005,9007199254740997,<NA>,None
5,006,9007199254740998,True,2026-01-10


## 2. Deux blocs et quatre situations

Le premier bloc demande `actif == True`. Le second demande une position strictement intérieure à la réunion du masque. La frontière ne satisfait pas `within`.

In [3]:
criteria = [{'field':'actif', 'comparison':'eq', 'value':True}]
selected = cm.extract_features(source, filters=criteria, mask=mask, predicate='within', id_field='code')
assert selected.data.code.tolist() == ['001']
assert selected.summary['combinations'] == {
    'both':1, 'attributes_only':2, 'spatial_only':1, 'neither':2}
assert len(selected.data) + len(selected.rejected) == len(source)
selected.report[['source_row','source_id','attribute_match','spatial_match','selected','reason']]

,source_row,source_id,attribute_match,spatial_match,selected,reason
0,0,001,True,True,True,selected
1,1,NA,True,False,False,spatial_no_match
2,2,003,False,True,False,attribute_no_match
3,3,004,False,False,False,attribute_and_spatial_no_match
4,4,005,False,False,False,attribute_and_spatial_no_match
5,5,006,True,False,False,spatial_no_match


## 3. OU et inversion finale

OU retient une ligne qui satisfait au moins un bloc. L’inversion prend ensuite le complément dans la source entière, y compris les géométries manquantes. Les décisions de départ restent dans le rapport.

In [4]:
either = cm.extract_features(source, filters=criteria, mask=mask, predicate='within', combination='or')
reverse = cm.extract_features(source, filters=criteria, mask=mask, predicate='within', combination='or', invert=True)
assert either.data.code.tolist() == ['001','NA','003','006']
assert reverse.data.code.tolist() == ['004','005']
assert either.summary['output_features'] + reverse.summary['output_features'] == len(source)
covered = cm.extract_features(source, mask=mask, predicate='covered_by')
assert covered.data.code.tolist() == ['001','003','006']
pd.DataFrame([{'mode':'ET', 'retenus':len(selected.data), 'exclus':len(selected.rejected)},
              {'mode':'OU', 'retenus':len(either.data), 'exclus':len(either.rejected)},
              {'mode':'Inverse de OU', 'retenus':len(reverse.data), 'exclus':len(reverse.rejected)}])

,mode,retenus,exclus
0,ET,1,5
1,OU,4,2
2,Inverse de OU,2,4


## 4. Dates natives, attributs multiples et valeurs manquantes

`all` et `any` combinent les critères à l’intérieur du bloc attributaire. Une date ne devient pas un texte dans les couches livrées. Un critère `is_null` cible les valeurs manquantes explicitement.

In [5]:
recent = cm.extract_features(source, filters=[
    {'field':'visite', 'comparison':'ge', 'value':'2026-02-01'},
    {'field':'actif', 'comparison':'eq', 'value':True},
], attribute_match='all')
assert recent.data.code.tolist() == ['NA']
missing = cm.extract_features(source, filters=[{'field':'visite','comparison':'is_null'}])
assert missing.data.code.tolist() == ['003','005']
exact = cm.select_attributes(source, 'identifiant', 'eq', str(2**53+1))
assert exact.code.tolist() == ['001']
recent.report[['source_id','attribute_1_match','attribute_2_match','reason']]

,source_id,attribute_1_match,attribute_2_match,reason
0,0,False,True,attribute_no_match
1,1,True,True,selected
2,2,False,False,attribute_no_match
3,3,True,False,attribute_no_match
4,4,False,False,attribute_no_match
5,5,False,True,attribute_no_match


## 5. Livraison vérifiée et audit réouvrable

La publication utilise un dossier neuf. Les retenus et les exclus possèdent les mêmes attributs ; leurs motifs sont dans le rapport. `source_row`, `output_row` et `rejected_row` commencent à zéro.

In [6]:
source_path = work/'visites.gpkg'
mask_path = work/'masque.gpkg'
write_geopackage(source, source_path, layer='visites')
write_geopackage(mask, mask_path, layer='emprise')
digests = {p:hashlib.sha256(p.read_bytes()).hexdigest() for p in (source_path,mask_path)}
parameters = {'data':str(source_path), 'layer':'visites', 'filters':criteria,
              'mask':str(mask_path), 'mask_layer':'emprise', 'predicate':'within', 'id_field':'code'}
delivery = cm.execute_operation('vector.extract_features', parameters, work/'livraison')
assert cm.read_file(delivery['products']['data']).code.tolist() == ['001']
assert len(cm.read_file(delivery['products']['rejected'])) == 5
report = table_page(delivery['products']['report'])['frame']
assert report.source_id.tolist() == source.code.tolist()
report[['source_row','source_id','output_row','rejected_row','selected','reason']]

,source_row,source_id,output_row,rejected_row,selected,reason
0,0,001,0,<NA>,True,selected
1,1,NA,<NA>,0,False,spatial_no_match
2,2,003,<NA>,1,False,attribute_no_match
3,3,004,<NA>,2,False,attribute_and_spatial_no_match
4,4,005,<NA>,3,False,attribute_and_spatial_no_match
5,5,006,<NA>,4,False,spatial_no_match


In [7]:
for name in ('data','rejected'):
    with sqlite3.connect(delivery['products'][name]) as db:
        schema = {r[1]:r[2] for r in db.execute('PRAGMA table_info(features)')}
        assert schema['visite'] == 'DATE'
        rows = db.execute('SELECT identifiant, code, visite FROM features ORDER BY rowid').fetchall()
        print(name, rows)
audit = json.loads(Path(delivery['products']['audit']).read_text(encoding='utf-8'))
assert audit['records'][0]['source_id'] == '001'
assert audit['records'][1]['source_id'] == 'NA'
assert audit['records'][0]['rejected_row'] is None

data [(9007199254740993, '001', '2026-01-01')]
rejected [(9007199254740994, 'NA', '2026-02-01'), (9007199254740995, '003', None), (9007199254740996, '004', '2026-03-01'), (9007199254740997, '005', None), (9007199254740998, '006', '2026-01-10')]


## 6. Réutiliser les exclus dans une chaîne

Un contrôle ultérieur peut porter sur les exclus, par exemple pour retrouver les dossiers incomplets. Ce second traitement ne modifie pas la première livraison.

In [8]:
plan = cm.processing_plan([
    {'id':'selection', 'operation':'vector.extract_features', 'parameters':parameters},
    {'id':'a_completer', 'operation':'vector.extract_features', 'parameters':{
        'data':'@selection:rejected', 'filters':[{'field':'visite','comparison':'is_null'}]}},
])
manifest = cm.run_plan(plan, work/'chaine')
chain = json.loads(Path(manifest).read_text(encoding='utf-8'))
assert cm.read_file(chain['products']['a_completer']['data']).code.tolist() == ['003','005']
print('Dossiers sans date parmi les exclus : 003, 005')

Dossiers sans date parmi les exclus : 003, 005

## 7. Une annulation ne livre rien

L’événement d’annulation est contrôlé pendant le calcul et avant publication. La vérification des sources s’appuie sur leurs empreintes exactes.

In [9]:
cancel = Event(); cancel.set()
try:
    cm.execute_operation('vector.extract_features', parameters, work/'annule', cancel=cancel)
except cm.ProcessingCancelled:
    pass
else:
    raise AssertionError('L’annulation devait interrompre le traitement')
assert not (work/'annule').exists()
assert source.geometry.to_wkb().tolist() == original_wkb
assert all(hashlib.sha256(p.read_bytes()).hexdigest() == digest for p,digest in digests.items())
assert not list(work.glob('.cartomize-*'))
print('Sources conservées, partitions vérifiées, annulation sans résultat partiel.')

Sources conservées, partitions vérifiées, annulation sans résultat partiel.


## Dans la fenêtre et le terminal

Ouvrir **Vecteur → Extraction avec bilan**, choisir la couche, ajouter les critères et activer éventuellement le masque. Les combinaisons et l’inversion reprennent les mêmes règles. Les boutons ouvrent les retenus, les exclus et les décisions.

Depuis le terminal : `cartomize extract-features visites.gpkg resultats/extraction_01 --layer visites --filters criteres.json`.

**À vérifier dans votre étude :** le sens du prédicat, la frontière, les valeurs manquantes et les quatre effectifs. Les opérations sont planaires ; le masque est aligné dans le SCR de la source. La qualité géométrique et les choix de reprojection se préparent avec G01/G04 lorsque nécessaire.